# Deploy Llama-3-8B-Instruct on SageMaker with SGLang DLC

This notebook deploys a Llama-3-8B-Instruct model using:
- **AWS Deep Learning Container** for **SGLang** (`sglang:0.5.10-gpu-py312-cu129-ubuntu24.04-sagemaker`)
- Model artifacts stored on **S3** (extracted to `/opt/ml/model`)
- Pure **boto3** calls (no sagemaker SDK dependency)

> **Container internals:** The `sagemaker_entrypoint.sh` converts `SM_SGLANG_*` env vars to `--cli-args` and runs `python3 -m sglang.launch_server`.  
> **Source:** [aws/deep-learning-containers](https://github.com/aws/deep-learning-containers/tree/master/sglang)  
> **DLC Images:** https://aws.github.io/deep-learning-containers/reference/available_images/

## 1. Setup & Configuration

In [1]:
import boto3
import json
import time

REGION = "us-east-1"
account_id = boto3.client("sts").get_caller_identity()["Account"]
role = "arn:aws:iam::129302905032:role/service-role/AmazonSageMaker-ExecutionRole-20260424T195813"

S3_MODEL_URI = "s3://sagemaker-us-east-1-129302905032/models/meta-llama-Meta-Llama-3-8B-Instruct/"
INSTANCE_TYPE = "ml.g5.2xlarge"
TENSOR_PARALLEL_DEGREE = 1

SGLANG_IMAGE_TAG = "0.5.10-gpu-py312-cu129-ubuntu24.04-sagemaker"
DLC_ACCOUNT_ID = "763104351884"
IMAGE_URI = f"{DLC_ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/sglang:{SGLANG_IMAGE_TAG}"

sm_client = boto3.client("sagemaker", region_name=REGION)
smr_client = boto3.client("sagemaker-runtime", region_name=REGION)

print(f"Account ID : {account_id}")
print(f"Region     : {REGION}")
print(f"Role       : {role}")
print(f"Model S3   : {S3_MODEL_URI}")
print(f"Image URI  : {IMAGE_URI}")
print(f"Instance   : {INSTANCE_TYPE}")

Account ID : 129302905032
Region     : us-east-1
Role       : arn:aws:iam::129302905032:role/service-role/AmazonSageMaker-ExecutionRole-20260424T195813
Model S3   : s3://sagemaker-us-east-1-129302905032/models/meta-llama-Meta-Llama-3-8B-Instruct/
Image URI  : 763104351884.dkr.ecr.us-east-1.amazonaws.com/sglang:0.5.10-gpu-py312-cu129-ubuntu24.04-sagemaker
Instance   : ml.g6e.2xlarge


## 2. Create SageMaker Model

In [4]:
model_name = f"meta-llama-Meta-Llama-3-8B-Instruct-{int(time.time())}"

env = {
    "SM_SGLANG_TP_SIZE": str(TENSOR_PARALLEL_DEGREE),
    "SM_SGLANG_DTYPE": "float16",
    "SM_SGLANG_CONTEXT_LENGTH": "8192",
    "SM_SGLANG_SERVED_MODEL_NAME": "Meta-Llama-3-8B-Instruct",
}

sm_client.create_model(
    ModelName=model_name,
    PrimaryContainer={
        "Image": IMAGE_URI,
        "ModelDataSource": {
            "S3DataSource": {
                "S3Uri": S3_MODEL_URI,
                "S3DataType": "S3Prefix",
                "CompressionType": "None",
            }
        },
        "Environment": env,
    },
    ExecutionRoleArn=role,
)

print(f"Created model: {model_name}")
print("\nSGLang CLI args derived from env:")
for k, v in env.items():
    cli_arg = "--" + k.replace("SM_SGLANG_", "").lower().replace("_", "-")
    print(f"  {k} = {v}  →  {cli_arg} {v}")

Created model: meta-llama-Meta-Llama-3-8B-Instruct-1777376275

SGLang CLI args derived from env:
  SM_SGLANG_TP_SIZE = 1  →  --tp-size 1
  SM_SGLANG_DTYPE = float16  →  --dtype float16
  SM_SGLANG_CONTEXT_LENGTH = 8192  →  --context-length 8192
  SM_SGLANG_SERVED_MODEL_NAME = Meta-Llama-3-8B-Instruct  →  --served-model-name Meta-Llama-3-8B-Instruct


## 3. Create Endpoint Config & Deploy

In [5]:
endpoint_config_name = f"llama-3-8b-sglang-epc-{int(time.time())}"
endpoint_name = f"llama-3-8b-sglang-ep-{int(time.time())}"

sm_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "InitialInstanceCount": 1,
            "InstanceType": INSTANCE_TYPE,
            "ContainerStartupHealthCheckTimeoutInSeconds": 1800,
            "ModelDataDownloadTimeoutInSeconds": 1800,
            "InferenceAmiVersion": "al2-ami-sagemaker-inference-gpu-3-1",
        }
    ],
)
print(f"Created endpoint config: {endpoint_config_name}")

sm_client.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name,
)
print(f"Creating endpoint: {endpoint_name} ...")
print("This takes ~10-15 minutes. Run next cell to wait.")

Created endpoint config: llama-3-8b-sglang-epc-1777376378
Creating endpoint: llama-3-8b-sglang-ep-1777376378 ...
This takes ~10-15 minutes. Run next cell to wait.


## 4. Wait for Endpoint to be InService

In [2]:
endpoint_name = f"llama-3-8b-sglang-ep-1777376378"
waiter = sm_client.get_waiter("endpoint_in_service")
waiter.wait(EndpointName=endpoint_name, WaiterConfig={"Delay": 30, "MaxAttempts": 60})

status = sm_client.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
print(f"Endpoint {endpoint_name} is now: {status}")

Endpoint llama-3-8b-sglang-ep-1777376378 is now: InService


## 5. Test Inference

In [5]:
payload = {
    "model": "llama3-8b",
    "messages": [
        {"role": "user", "content": "What is the capital of France?"}
    ],
    "max_tokens": 256,
    "temperature": 0.7,
    "top_p": 0.9,
}

response = smr_client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=json.dumps(payload),
)

result = json.loads(response["Body"].read().decode("utf-8"))
print("=== Model Response ===")
print(json.dumps(result, indent=2, ensure_ascii=False))

=== Model Response ===
{
  "id": "7b5b1960e54b4919a4ab456abc09e4cd",
  "object": "chat.completion",
  "created": 1777379155,
  "model": "llama3-8b",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "The capital of France is Paris.",
        "reasoning_content": null,
        "tool_calls": null
      },
      "logprobs": null,
      "finish_reason": "stop",
      "matched_stop": 128009
    }
  ],
  "usage": {
    "prompt_tokens": 17,
    "total_tokens": 25,
    "completion_tokens": 8,
    "prompt_tokens_details": null,
    "reasoning_tokens": 0
  },
  "metadata": {
    "weight_version": "default"
  }
}


In [5]:
messages = [{
        "role": "user",
        "content": "What is the capital of France?"
}]
payload = {
    "messages": messages,
    "max_tokens": 1024,
    "stream": False
}
response = smr_client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType='application/json',
    Body=json.dumps(payload)
)

print(json.loads(response['Body'].read())["choices"][0]["message"]["content"])

{'id': '4fb11887be264e7f947a85eded6a1165', 'object': 'chat.completion', 'created': 1777432980, 'model': 'default', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'The capital of France is Paris.', 'reasoning_content': None, 'tool_calls': None}, 'logprobs': None, 'finish_reason': 'stop', 'matched_stop': 128009}], 'usage': {'prompt_tokens': 17, 'total_tokens': 25, 'completion_tokens': 8, 'prompt_tokens_details': None, 'reasoning_tokens': 0}, 'metadata': {'weight_version': 'default'}}


## 6. Test Streaming Inference (Optional)

In [ ]:
payload_stream = {
    "model": "llama3-8b",
    "messages": [
        {"role": "user", "content": "Explain quantum computing in simple terms."}
    ],
    "max_tokens": 256,
    "temperature": 0.7,
    "stream": True,
}

response_stream = smr_client.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=json.dumps(payload_stream),
)

print("=== Streaming Response ===")
event_stream = response_stream["Body"]
for event in event_stream:
    chunk = event.get("PayloadPart", {}).get("Bytes", b"")
    if chunk:
        text = chunk.decode("utf-8")
        print(text, end="", flush=True)
print()

## 7. Cleanup

**Run this cell to delete the endpoint and avoid ongoing charges.**

In [ ]:
sm_client.delete_endpoint(EndpointName=endpoint_name)
print(f"Deleted endpoint: {endpoint_name}")

sm_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
print(f"Deleted endpoint config: {endpoint_config_name}")

sm_client.delete_model(ModelName=model_name)
print(f"Deleted model: {model_name}")

print("\nCleanup complete!")